# Submit 44 (ENTREGA FINAL) — juntar con hiperparámetros de z028, top 12.000. Código completo de principio a fin

**Resultado:** `44_ens70juntarZ028_12000.csv` = los 12.000 clientes de 202108 con mayor probabilidad promedio de 70 modelos LightGBM.
Es copia exacta del submit 39 (público 104,69), reenviado como último submit para que quede como entrega.

**Receta**
1. Datos: `competencia_01.parquet`, creado con `datos_crear_parquet.ipynb` (clase_ternaria estricta; `ccajas_depositos` = NA en 202105).
2. Variables: las 153 originales (incluye `numero_de_cliente`) + lags y deltas de orden 1 y 2 = **761 variables**.
3. Target "juntar": 1 si el cliente es BAJA+1 o BAJA+2, 0 si es CONTINUA.
4. Train: 202103, 202104, 202105 y 202106 completos (sin undersampling).
5. Modelos: 7 trials de la optimización bayesiana z028 (`0_exploracion/z028_optuna_juntar.ipynb`: Optuna con target juntar,
   5 semillas por trial, revalidados con 10 semillas en junio y mayo) × 10 semillas = **70 modelos**.
   Hiperparámetros fijos del profesor: learning_rate 0,005, min_data_in_leaf 0, max_bin 31, feature_fraction 0,5, feature_fraction_bynode 0,2.
6. Ensamble: promedio simple de las 70 probabilidades por cliente → top 12.000.

**Historia:** los 70 modelos se entrenaron con `39-43_ens70juntarZ028_completo.ipynb` (mismo código, 400 min). Este notebook
reusa esas predicciones (copiadas a `probs_44/`) y genera el archivo final. Reproducibilidad verificada: un modelo reentrenado
en un proceso nuevo con este código da una predicción idéntica (`np.array_equal` = True).

**Tiempo:** ~6,7 h desde cero (~5,7 min por modelo). Cada predicción se guarda en `probs_44/`: si se corta, volver a correr y retoma.


In [1]:
# --- 1. Librerías y parámetros ---
import os
import time
import numpy as np
import pandas as pd
import lightgbm as lgb

MESES_FINAL = [202103, 202104, 202105, 202106]   # meses de entrenamiento
MES_PRED    = 202108                             # mes a predecir
ENVIOS      = {44: 12_000}                       # número de submit -> cantidad de clientes
CARPETA     = "probs_44"                         # una predicción por modelo (para retomar si se corta)
os.makedirs(CARPETA, exist_ok=True)

# Las 10 semillas del 32, en los mismos dos grupos. Cada grupo usa su propio Dataset (ver celda 3).
GRUPOS_SEMILLAS = {
    "submit15": [501419, 623353, 786349, 834569, 946331],
    "submit16": [270131, 417293, 583117, 692347, 851071],
}

# Hiperparámetros fijos (recomendación del profesor)
params_fijos = {
    "objective": "binary", "learning_rate": 0.005, "min_data_in_leaf": 0,
    "feature_fraction": 0.5, "feature_fraction_bynode": 0.2, "max_bin": 31, "verbose": -1,
}

# Los 7 trials de z028 elegidos: trial -> (num_leaves, min_sum_hessian_in_leaf, cantidad de árboles)
# (copiados de 0_exploracion/z028_optuna_trials.csv, con todos los decimales; árboles = promedio del early stopping de 5 semillas)
TRIALS = {
    16: (64, 6.982303130196578, 1420),
    18: (47, 7.044084165759745, 1525),
    34: (65, 9.553255512602105, 1427),
    15: (42, 9.65435654722589, 1532),
    21: (38, 5.046242592824922, 1660),
    36: (106, 4.026506925773964, 1248),
    32: (53, 5.459562821619835, 1443),
}

In [2]:
# --- 2. Datos, lags/deltas de orden 1 y 2, y target juntar ---
df = pd.read_parquet("../competencia_01.parquet")

no_usar = ["foto_mes", "clase_ternaria"]
cols = [c for c in df.columns if c not in no_usar and c != "numero_de_cliente"]   # a estas se les calculan lags

# Ordeno por cliente y mes para que shift(k) traiga el valor de k meses antes del MISMO cliente
df = df.sort_values(["numero_de_cliente", "foto_mes"]).reset_index(drop=True)
grupo = df.groupby("numero_de_cliente")[cols]
nuevas = []
for k in [1, 2]:
    lag = grupo.shift(k).astype("float32")              # valor de hace k meses
    delta = (df[cols] - lag).astype("float32")          # cuánto cambió desde entonces
    lag.columns   = [f"{c}_lag{k}" for c in cols]
    delta.columns = [f"{c}_delta{k}" for c in cols]
    nuevas += [lag, delta]
df = pd.concat([df] + nuevas, axis=1)
del nuevas, lag, delta, grupo

variables = [c for c in df.columns if c not in no_usar]   # 761, incluye numero_de_cliente
final  = df[df["foto_mes"].isin(MESES_FINAL)]
y_final = final["clase_ternaria"].isin(["BAJA+1", "BAJA+2"]).astype(int)   # target juntar
futuro = df[df["foto_mes"] == MES_PRED][["numero_de_cliente"]].copy()
X_futuro = df.loc[futuro.index, variables]
print("Variables:", len(variables), "| filas train:", len(final), "| positivos:", y_final.sum())

Variables: 761 | filas train: 654066 | positivos: 8067


In [3]:
# --- 3. Funciones ---
def armar_dataset(semilla_grupo):
    # LightGBM arma los "bines" de cada variable (max_bin = 31) con una muestra elegida con la semilla.
    # Igual que en el 32: un Dataset por grupo de semillas, armado con la primera semilla del grupo.
    datos = lgb.Dataset(final[variables], label=y_final, free_raw_data=False,
                        params={**params_fijos, "seed": semilla_grupo})
    return datos.construct()

def entrenar_y_predecir(datos, trial, semilla):
    hojas, hessian, arboles = TRIALS[trial]
    p = {**params_fijos, "num_leaves": hojas, "min_sum_hessian_in_leaf": hessian, "seed": semilla}
    modelo = lgb.train(p, datos, num_boost_round=arboles)
    return modelo.predict(X_futuro)                  # probabilidad de baja de cada cliente de 202108

In [4]:
# --- 4. Entrenar los 70 modelos (retoma si se cortó) ---
inicio_total = time.time()
for nombre, semillas in GRUPOS_SEMILLAS.items():
    if all(os.path.exists(f"{CARPETA}/t{t}_s{s}.npy") for t in TRIALS for s in semillas):
        continue                                     # grupo completo: no hace falta armar el Dataset
    datos = armar_dataset(semillas[0])
    for trial in TRIALS:
        for semilla in semillas:
            archivo = f"{CARPETA}/t{trial}_s{semilla}.npy"
            if os.path.exists(archivo):              # ya estaba hecho: lo salteamos
                continue
            inicio = time.time()
            np.save(archivo, entrenar_y_predecir(datos, trial, semilla))
            with open("44_progreso.txt", "a", encoding="utf-8") as f:
                f.write(time.strftime("%H:%M:%S ") + f"{nombre} | trial {trial}, semilla {semilla}: {(time.time() - inicio) / 60:.1f} min\n")
    del datos
print(f"Total: {(time.time() - inicio_total) / 60:.0f} min")

Total: 0 min


In [5]:
# --- 5. Ensamble (promedio de las 70 probabilidades) y un archivo por corte ---
todas = [s for semillas in GRUPOS_SEMILLAS.values() for s in semillas]
probs = [np.load(f"{CARPETA}/t{t}_s{s}.npy") for t in TRIALS for s in todas]
futuro["prob_ensamble"] = np.mean(probs, axis=0)
print("Modelos en el ensamble:", len(probs))
futuro.to_parquet("44_ens70_probs.parquet")
orden = futuro.sort_values("prob_ensamble", ascending=False)["numero_de_cliente"].astype("int64")

entrega_32 = set(pd.read_csv("32_ens100juntar_12000.csv", header=None)[0])
for numero, n in ENVIOS.items():
    archivo = f"{numero}_ens70juntarZ028_{n}.csv"
    top = orden.head(n)
    top.to_csv(archivo, index=False, header=False)     # una columna, sin encabezado (formato del bot)
    contenido = open(archivo).read()
    print(f"{archivo} | filas: {len(contenido.splitlines())} | únicos: {top.nunique()} | ¿e+? {'e+' in contenido}"
          f" | coincide con el 32: {len(entrega_32 & set(top)) / len(entrega_32):.1%}")

Modelos en el ensamble: 70


44_ens70juntarZ028_12000.csv | filas: 12000 | únicos: 12000 | ¿e+? False | coincide con el 32: 94.2%


In [6]:
# --- 6. Verificación: ¿es el mismo archivo que el submit 39? ---
iguales = open("44_ens70juntarZ028_12000.csv", "rb").read() == open("39_ens70juntarZ028_12000.csv", "rb").read()
print("¿44 idéntico byte a byte al 39?", iguales)

¿44 idéntico byte a byte al 39? True
